In [18]:
# 1 . Connect to db
import sqlite3
import pandas as pd 
import os 

os.chdir (r'D:\Mission_Blitzkreig\Month_2_SQL')

conn = sqlite3.connect('marketing_campaigns.db')

cursor = conn.cursor()
print("Connected to the database")

Connected to the database


In [19]:
# 2. Sub-queries in "Where" clause
query_where_subquery = """    
SELECT
      campaign_name,
      city,
      spend
FROM campaigns
WHERE spend > (SELECT AVG(spend) FROM campaigns)
ORDER BY spend DESC
"""

df_above_avg = pd.read_sql_query(query_where_subquery, conn)
print("\n CAMPAIGNS WITH ABOVE AVERAGE SPEND:")
print (df_above_avg)
print(f"\n Found {len(df_above_avg)} campaigns above average")


 CAMPAIGNS WITH ABOVE AVERAGE SPEND:
                   campaign_name       city  spend
0     Mumbai Airport - Supersite     Mumbai   8500
1  Chennai City Hall - Supersite    Chennai   6200
2       Mumbai Centre - 48 Sheet     Mumbai   4200
3       Bangalore City - Digital  Bangalore   3400

 Found 4 campaigns above average


In [20]:
# 3. Sub-queries in "Where" (using IN)
query_in_subquery = """ 
SELECT
     campaign_name,
     city,
     format
FROM campaigns
WHERE city 
IN (SELECT DISTINCT city FROM campaigns WHERE format LIKE '%48 sheet%') 
ORDER BY city,campaign_name
"""

df_bus_cities = pd.read_sql_query(query_in_subquery, conn)

print("\n All campaigns in cities with BUS format :")
print(df_bus_cities)

print(f"\n Found {len(df_bus_cities)} campaigns with bus formats")



 All campaigns in cities with BUS format :
                campaign_name    city     format
0    Delhi Motorway - Digital   Delhi    Digital
1     Delhi Retail - 48 Sheet   Delhi   48 Sheet
2  Mumbai Airport - Supersite  Mumbai  Supersite
3    Mumbai Centre - 48 Sheet  Mumbai   48 Sheet
4       Mumbai DART - 4 Sheet  Mumbai    4 Sheet

 Found 5 campaigns with bus formats


In [21]:
#4.Sub-queries in SELECT clause 
query_select_subquery = """
SELECT
       campaign_name,
       city,
       spend,
       (SELECT AVG(spend) FROM campaigns) AS avg_spend,spend - (SELECT  AVG(spend) FROM campaigns)AS spend_diff
       FROM campaigns 
       ORDER BY spend DESC 
       LIMIT 10 
       """

df_spend_comparision = pd.read_sql_query(query_select_subquery, conn)
print("\n CAMPAIGN SPEND VS AVERAGE SPEND:")
print(df_spend_comparision)


 CAMPAIGN SPEND VS AVERAGE SPEND:
                   campaign_name       city  spend  avg_spend  spend_diff
0     Mumbai Airport - Supersite     Mumbai   8500     3365.0      5135.0
1  Chennai City Hall - Supersite    Chennai   6200     3365.0      2835.0
2       Mumbai Centre - 48 Sheet     Mumbai   4200     3365.0       835.0
3       Bangalore City - Digital  Bangalore   3400     3365.0        35.0
4        Delhi Retail - 48 Sheet      Delhi   3100     3365.0      -265.0
5       Delhi Motorway - Digital      Delhi   2800     3365.0      -565.0
6          Mumbai DART - 4 Sheet     Mumbai   1800     3365.0     -1565.0
7     Chennai Roadside - 6 Sheet    Chennai   1500     3365.0     -1865.0
8      Goa Retail Park - 6 Sheet        Goa   1200     3365.0     -2165.0
9        Pune City - Bus Shelter       Pune    950     3365.0     -2415.0


In [22]:
# 5. Subqueries in FROM clause 
query_from_subquery = """
SELECT
      city,
      COUNT(*) as campaign_count,
      SUM(spend) as total_spend,
      AVG(conversions) as avg_conversions
FROM (
    SELECT
        city,
        spend,
        conversions
    FROM campaigns
    WHERE spend > 5000
)
GROUP BY city
ORDER BY total_spend DESC 
""" 

df_city_summary = pd.read_sql_query(query_from_subquery, conn)

print("\n CITY SUMMARY WITH SPEND > 50k ")
print(df_city_summary)


 CITY SUMMARY WITH SPEND > 50k 
      city  campaign_count  total_spend  avg_conversions
0   Mumbai               1         8500            210.0
1  Chennai               1         6200            148.0


In [23]:
#6. Correlated subquery (references outer query)
query_correlated = """
SELECT 
    campaign_name,  -- Campaign name
    city,  -- City where campaign ran
    spend,  -- Spend on this campaign
    (SELECT AVG(spend) FROM campaigns c2 WHERE c2.city = c1.city) AS city_avg_spend
    -- Subquery that uses city from outer query (c1.city)
FROM campaigns c1  -- Alias c1 for outer campaigns table
WHERE spend > (  -- Filter to campaigns with spend greater than...
    SELECT AVG(spend)  -- Average spend for...
    FROM campaigns c2
    WHERE c2.city = c1.city  -- That specific city (correlated)
)
ORDER BY city, spend DESC  -- Sort by city, then spend highest first
"""

# Execute and display results
df_above_city_avg = pd.read_sql_query(query_correlated, conn)
print("\n📊 Campaigns spending more than their city's average:")
print(df_above_city_avg)
print("\n✅ Correlated subquery compares each campaign to ITS city's average")


📊 Campaigns spending more than their city's average:
                   campaign_name     city  spend  city_avg_spend
0  Chennai City Hall - Supersite  Chennai   6200     3850.000000
1        Delhi Retail - 48 Sheet    Delhi   3100     2950.000000
2     Mumbai Airport - Supersite   Mumbai   8500     4833.333333

✅ Correlated subquery compares each campaign to ITS city's average


In [24]:
# 7. Sub-queries VS joins 
# Approach 1 : Subqueries 
query_app1 = """
SELECT 
      campaign_name,
      city,
      spend
FROM campaigns
WHERE city IN (
     SELECT city FROM campaigns WHERE conversions > 100)
ORDER BY city 
"""  

# Approach 2 : JOINS 
query_app2 = """ 
SELECT DISTINCT 
               c1.campaign_name,
               c1.city,
               c1.spend
FROM campaigns c1 
INNER JOIN campaigns c2 ON c1.city = c2.city 
WHERE c2 .conversions > 100
ORDER BY c1.city
"""

print(f"\n Approach 1 : Sub-queries")
df_app1 = pd.read_sql_query(query_app1, conn)
print(df_app1)

print(f"\n Approach 2 : JOINS")
df_app2 = pd.read_sql_query(query_app2, conn)
print(df_app2)



 Approach 1 : Sub-queries
                   campaign_name     city  spend
0     Chennai Roadside - 6 Sheet  Chennai   1500
1  Chennai City Hall - Supersite  Chennai   6200
2       Mumbai Centre - 48 Sheet   Mumbai   4200
3     Mumbai Airport - Supersite   Mumbai   8500
4          Mumbai DART - 4 Sheet   Mumbai   1800

 Approach 2 : JOINS
                   campaign_name     city  spend
0  Chennai City Hall - Supersite  Chennai   6200
1     Chennai Roadside - 6 Sheet  Chennai   1500
2     Mumbai Airport - Supersite   Mumbai   8500
3       Mumbai Centre - 48 Sheet   Mumbai   4200
4          Mumbai DART - 4 Sheet   Mumbai   1800


In [30]:
# PRACTICE : 
#Ex.1 Find above avg conversions 
query_above_average = """ 
SELECT 
      campaign_name,
      city,
      conversions
FROM campaigns
WHERE conversions > (SELECT AVG(conversions) FROM campaigns)
ORDER BY conversions DESC 
""" 
df_1 = pd.read_sql_query(query_above_average, conn)
print(df_1)

#Ex.2 City with highest avg spend
query_2 = """ 
SELECT 
      city, 
      AVG(spend) as avg_spend
FROM campaigns 
GROUP BY city
ORDER BY avg_spend DESC
LIMIT 1 
"""

df_2 = pd.read_sql_query(query_2, conn)
print("\n HIGHEST AVERAGE SPEND CITY")
print(df_2)


# 3.Show each campaign spend vs overall spend 
query_3 = """ 
SELECT 
      campaign_name, 
      spend,
      ( SELECT AVG(spend) FROM campaigns) as overall_avg,
      spend - (SELECT AVG(spend) FROM campaigns) as difference
FROM campaigns
ORDER BY spend DESC
LIMIT 5
"""
df_3 = pd.read_sql_query(query_3, conn)
print("\n CAMPAIGN SPEND VS OVERALL SPEND")
print(df_3)

#4.Campaigns in city where sum of spend >10000
query_4 = """  
SELECT 
       campaign_name,
       city,
       spend
FROM campaigns
WHERE city IN (
       SELECT city
       FROM campaigns
       Group BY city
       HAVING SUM (spend) > 10000)
ORDER BY city , spend DESC 
"""
df_4 = pd.read_sql_query(query_4, conn)
print("\n Campaigns in city with high spend")
print(df_4)

cursor.close()
conn.close()



                   campaign_name       city  conversions
0     Mumbai Airport - Supersite     Mumbai          210
1  Chennai City Hall - Supersite    Chennai          148
2       Mumbai Centre - 48 Sheet     Mumbai           87
3       Bangalore City - Digital  Bangalore           79

 HIGHEST AVERAGE SPEND CITY
     city    avg_spend
0  Mumbai  4833.333333

 CAMPAIGN SPEND VS OVERALL SPEND
                   campaign_name  spend  overall_avg  difference
0     Mumbai Airport - Supersite   8500       3365.0      5135.0
1  Chennai City Hall - Supersite   6200       3365.0      2835.0
2       Mumbai Centre - 48 Sheet   4200       3365.0       835.0
3       Bangalore City - Digital   3400       3365.0        35.0
4        Delhi Retail - 48 Sheet   3100       3365.0      -265.0

 Campaigns in city with high spend
                campaign_name    city  spend
0  Mumbai Airport - Supersite  Mumbai   8500
1    Mumbai Centre - 48 Sheet  Mumbai   4200
2       Mumbai DART - 4 Sheet  Mumbai   1800
